In [1]:
# Load model directly
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
import torch
import os
from pdf2image import convert_from_path
from PIL import Image, ImageChops
import datetime



In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")
model = Qwen3VLForConditionalGeneration.from_pretrained("Qwen/Qwen3-VL-2B-Instruct",
                                                        dtype=torch.bfloat16,
                                                        device_map="auto",
                                                        # attn_implementation="flash_attention_2"
                                                        )



Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [3]:
directory_name = "resume_samples_pdf"
resume_img_directory = "resume_sample_imgs"
target_output_directory = "sample_output"

pdf_files = [f for f in os.listdir(directory_name) if f.endswith('.pdf') or f.endswith('.PDF')]

In [4]:
print(pdf_files)

['Afundar_AudrieLex_CV.pdf', 'Cuerdo_NaomiHannah-CV.pdf', 'Khafaji,Mostafa_Resume_2025.pdf', 'Percia_KyteDaiter_CV.pdf', 'Rodillas_ChristianMiguel_CV.pdf']


In [5]:
def trim_whitespace(im):
    bg = Image.new(im.mode, im.size, im.getpixel((0,0)))
    diff = ImageChops.difference(im, bg)
    diff = ImageChops.add(diff, diff, 2.0, -100)
    bbox = diff.getbbox()
    if bbox:
        return im.crop(bbox)
    return im

image_list = []
for pdf in pdf_files:
    file_path = os.path.join(directory_name, pdf)

    images = convert_from_path(file_path, dpi=150, fmt="png")
    images = [trim_whitespace(img) for img in images]

    for idx, image in enumerate(images):
        os.makedirs(fr"{resume_img_directory}/{pdf.rsplit('.', 1)[0]}", exist_ok = True)
        image.save(fr"{resume_img_directory}/{pdf.rsplit('.', 1)[0]}/{idx+1}.png", "PNG")

In [6]:
img_files = [f for f in os.listdir(resume_img_directory)]

os.makedirs(target_output_directory, exist_ok=True)

In [7]:
for file in img_files:
    print(file)

Afundar_AudrieLex_CV
Cuerdo_NaomiHannah-CV
Khafaji,Mostafa_Resume_2025
Modern Professional CV Resume.png
Percia_KyteDaiter_CV
Professional Minimalist CV Resume.png
random_sample.png
Rodillas_ChristianMiguel_CV


In [8]:
# Use a raw string with double-braces for the JSON structure
prompt = f"""
Extract and structure the information from the following Resume into a clean JSON format.

### INSTRUCTIONS ###
1. Clean up OCR errors and normalize text.
2. For 'Work Experience' and 'Education', concatenate all descriptions/details into a single coherent paragraph for each entry.
3. If information is missing, use an empty string "" or an empty list [].
4. Output ONLY valid JSON. Do not include introductory text.

### TARGET JSON STRUCTURE ###
{{
    "personal_info": {{
        "first_name": "",
        "last_name": "",
        "email": "",
        "phone": "",
        "address": "",
        "city": "",
        "country": ""
    }},
    "experiences": [
        {{
            "job_title": "",
            "company": "",
            "dates": "",
            "responsibilities": ""
        }}
    ],
    "education": [
        {{
            "degree": "",
            "institution": "",
            "graduation_date": "",
            "details": ""
        }}
    ],
    "skills": {{
        "technical": "",
        "soft": "",
        "interests": ""
    }}
}}
"""

In [ ]:
for file in img_files:

    file_path = os.path.join(resume_img_directory, file)

    img_paths = []
    if os.path.isdir(file_path):

        img_files = [f for f in os.listdir(file_path)]
        img_paths = [os.path.join(file_path, f) for f in img_files]
    else:
        img_files = [file]
        img_paths = [file_path]

    list_output = []
    for img_path in img_paths:

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "path":img_path},
                    {
                        "type": "text", "text": prompt}
                ]
            },
        ]

        time_start = datetime.datetime.now()
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        print(f"Generating Output for {file}...")
        outputs = model.generate(**inputs, max_new_tokens=512)

        print("Output Generated")
        json_output = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:])
        list_output.append(json_output)

        time_end = datetime.datetime.now()
        time_difference = (time_end - time_start)

        minutes = divmod(time_difference.total_seconds(), 60)
        print(f"Model Execution time for {img_path} is {minutes[0]} minutes and {minutes[1]} seconds.")

    # print(json_output)
    with open(f"{target_output_directory}/{file.rsplit('.', 1)[0]}.json", "w", encoding='utf-8') as f:
        f.write(",\n".join(list_output))
    print(f"Saved to sample_output/{file.rsplit('.', 1)[0]}.json")

Generating Output for Afundar_AudrieLex_CV...
Output Generated
Model Execution time for resume_sample_imgs\Afundar_AudrieLex_CV\1.png is 27.0 minutes and 56.23345900000004 seconds.
Generating Output for Afundar_AudrieLex_CV...
Output Generated
Model Execution time for resume_sample_imgs\Afundar_AudrieLex_CV\2.png is 20.0 minutes and 42.575824999999895 seconds.
Generating Output for Afundar_AudrieLex_CV...
Output Generated
Model Execution time for resume_sample_imgs\Afundar_AudrieLex_CV\3.png is 11.0 minutes and 42.71136999999999 seconds.
Saved to sample_output/Afundar_AudrieLex_CV.json
Generating Output for Cuerdo_NaomiHannah-CV...


In [ ]:
# # one resume sample
#
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "image", "path":"resume_sample_imgs/random_sample.png"},
#             {"type": "text", "text": """Parse the text from this resume and extract the following sections:
#                 - Personal Information
#                 - Work Experience (job titles, companies, dates, responsibilities)
#                 - Education (degrees, institutions, graduation dates)
#                 - Skills (technical skills, tools, certifications)
#
#                 Return the information in a structured JSON format."""}
#         ]
#     },
# ]
#
# inputs = processor.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_dict=True,
#     return_tensors="pt",
# ).to(model.device)
#
# print(f"Generating Output for sample random resume...")
# outputs = model.generate(**inputs, max_new_tokens=2048)
#
# print("Output Generated")
# json_output = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:])
#
# # print(json_output)
# with open(f"sample_output/random_sample_resume.json", "w", encoding='utf-8') as f:
#     f.write(json_output)
# print(f"Saved to sample_output/random_sample_resume.json")

In [ ]:
# inputs = processor.apply_chat_template(
# 	messages,
# 	add_generation_prompt=True,
# 	tokenize=True,
# 	return_dict=True,
# 	return_tensors="pt",
# ).to(model.device)
#
# outputs = model.generate(**inputs, max_new_tokens=3072)
# print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
#


In [ ]:
# with open("output.json", "w") as f:
#     json_output = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:])
#     f.write(json_output)